### Code developed by Susanne Ulmer and Mohammad Amin Eskandari

# Initialization

## Imports

In [1]:
# Standard Library
from pathlib import Path

# Third-Party Libraries
import numpy as np
import scipy

# Specialized External Libraries (Optimal Transport)
import LinOT
import LinOT.auxiliary as aux
import LinOT.SinkhornNP as Sinkhorn
import LinOT.Visualization as Vis
import MultiScaleOT

# Local Project Modules
import src.average_cell_functions as ac

## Required constants

In [2]:
# number of images to select
nImg = 50 # We have used 30 images for actin, microtubules and ER, while 50 images for Golgi and nucleus

structure_name = "nucleus" # replace with "endoplasmic reticulum", "golgi", "microtubules" or "actin" for other structures
structure_name_folder = structure_name.replace(" ", "_")

ds_factor = 2 # downsample factor

## Retrieve required file paths

In [3]:
base_dir = Path("./Example_data/Average_cell")
save_dir = base_dir / f"data_processed/{structure_name_folder}/"

## Loading the images

In [4]:
if not save_dir.exists():
    save_dir.mkdir()

raw_data = np.load(base_dir / "data" /  f"structures_images.npz")[structure_name]

# We load the indices used in the paper from npy file.
selected_indices = np.load(save_dir / f"{structure_name_folder}_indices.npy")

##############################################################################################################
####################### To select another set of images uncomment the following ##############################
##############################################################################################################


## to exclude images which are not segmented well
# remove_indices = [12, 13, 18, 35, 60, 68, 137, 152, 176, 190, 195, 198, 199, 200, 201, 206, 213, 215, 219, 222, 223, 230]

## For other structures, uncomment one of the following lines ###########################

# remove_indices = [5, 12, 26, 35] # ER
# remove_indices = [] # Actin, Golgi, Microtubules


## all_indices = np.arange(len(raw_data))
## valid_indices = np.setdiff1d(all_indices, remove_indices)

## Randomly sample from the VALID indices
## We use the valid_indices themselves as the choice pool
# nImg = min(nImg, len(valid_indices)) # Ensure we don't ask for more than exist
# selected_indices = np.random.choice(valid_indices, nImg, replace=False)

## to save the indices of new set, uncomment the following line
## np.save(save_dir / f"{structure_name_folder}_indices.npy", selected_indices)

##############################################################################################################
##############################################################################################################


# Load the data using these specific pointers
data = raw_data[selected_indices]

# Post-processing (Normalization)
sums = np.sum(data, axis=(1,2))
nonzero_mask = sums > 0

data = data[nonzero_mask]
# If any were zero, update our count
nImg = data.shape[0]

# Rescale by sums of the kept images
data = data / sums[nonzero_mask].reshape((-1, 1, 1))

print(f"Number of images: {nImg}")

Number of images: 50


# Calculating the Wasserstein barycenter

In [ ]:
print(
    "Calculating barycenter...\n"
    "Note: This step can take from 10+ minutes to several hours depending on\n"
    "      the image dimensions, level of detail, and total number of frames.\n"
    "Please wait..."
)

imgCenter = np.mean(data, axis=0)

measureList = [ac.extractMeasureFromImage(img, zeroThresh=1E-14) for img in data]
measureCenter = ac.extractMeasureFromImage(imgCenter, zeroThresh=1E-14)

nMarginals = len(measureList)

weightList = np.ones(nMarginals)
weightList = weightList / np.sum(weightList)

# determines how many layers the multiscale representation will have
hierarchyDepth = 7

# generate multi scale objects, do not allocate dual variable memory
MultiScaleSetupList = [MultiScaleOT.TMultiScaleSetup(pos, mu, hierarchyDepth,0,setupDuals=False) for mu,pos in measureList]

# generate center object
MultiScaleSetupCenter=MultiScaleOT.TMultiScaleSetup(measureCenter[1],measureCenter[0],hierarchyDepth,0,setupDuals=False)

nLayers=MultiScaleSetupCenter.getNLayers()

# list of cost function objects
CostFunctionList=[MultiScaleOT.THierarchicalCostFunctionProvider_SquaredEuclidean(multiX,MultiScaleSetupCenter)\
        for multiX in MultiScaleSetupList]

BarycenterContainer=MultiScaleOT.TMultiScaleSetupBarycenterContainer(nMarginals)

# assign multi scale objects to barycenter object
for i in range(nMarginals):
    BarycenterContainer.setMarginal(i,MultiScaleSetupList[i],weightList[i])

BarycenterContainer.setCenterMarginal(MultiScaleSetupCenter)


# now allocate dual variables for barycenter problem. the memory is managed by the
# TMultiScaleSetupBarycenterContainer object, not by the separate TMultiScaleSetup objects
BarycenterContainer.setupDuals()

# assign cost function objects to barycenter object
for i in range(nMarginals):
    BarycenterContainer.setCostFunctionProvider(i,CostFunctionList[i])

MultiScaleOT.setVerboseMode(False)

errorGoal=1E-3
cfg=MultiScaleOT.TSinkhornSolverParameters()
epsScalingHandler=MultiScaleOT.TEpsScalingHandler()
epsScalingHandler.setupGeometricMultiLayerB(nLayers,1.,4.,2,2)

SinkhornSolver=MultiScaleOT.TSinkhornSolverBarycenter(epsScalingHandler,0,hierarchyDepth,errorGoal,\
        BarycenterContainer,cfg)
SinkhornSolver.initialize()
SinkhornSolver.solve()
print("solved successfully")
print("dump to file.",flush=True)

alphaList=[BarycenterContainer.getAlpha(i,nLayers-1) for i in range(nMarginals)]
betaList=[BarycenterContainer.getBeta(i,nLayers-1) for i in range(nMarginals)]

alphaDict={"alpha"+str(i):alphaList[i] for i in range(nMarginals)}

MultiScaleOT.setVerboseMode(False)

eps=epsScalingHandler.get()[-1][-1]

SinkhornSolver.changeLayer(nLayers-1)
SinkhornSolver.changeEps(eps)

for i in range(nMarginals):
    BarycenterContainer.setAlpha(alphaList[i],i,nLayers-1)
    BarycenterContainer.setBeta(betaList[i],i,nLayers-1)

SinkhornSolver.generateKernel()

barycenterList=[]
barycenterImgList=[]
for i in range(nMarginals):
    kernelData=SinkhornSolver.getKernelCSRDataTuple(i)
    xres=len(alphaList[i])
    yres=len(betaList[i])
    kernel=scipy.sparse.csr_array(kernelData,shape=(xres,yres))
    barycenter=kernel.sum(axis=0)
    barycenterList.append(barycenter)

    container=MultiScaleOT.TParticleContainer(barycenter,measureCenter[1])
    imgBarycenter=np.zeros_like(imgCenter)
    MultiScaleOT.projectInterpolation(container,imgBarycenter)
    barycenterImgList.append(imgBarycenter)

barycenterList=np.array(barycenterList)
barycenterImgList=np.array(barycenterImgList)

## These lines save the barycenter

# output_path = save_dir / f"{structure_name}_barycenter.npz"
# np.savez(output_path,betaList=betaList,nMarginals=nMarginals,**alphaDict)


# Calculating tangent vectors

## Prepare measures

In [6]:
 # extract point mass data
sampleDat=[aux.processDensity_Grid(img, totalMass=1.,keepZero=False) for img in data]
muRef,posRef=aux.processDensity_Grid(imgCenter, totalMass=1., keepZero=False)

## Solve OT Maps

In [7]:
def getTan(i):
    hierarchyDepth=7
    nLayers=hierarchyDepth+1

    mu,pos=sampleDat[i]

    MultiCenter=MultiScaleOT.TMultiScaleSetup(posRef,muRef,hierarchyDepth,0)
    MultiX=MultiScaleOT.TMultiScaleSetup(pos,mu,hierarchyDepth,0)

    costFunction=MultiScaleOT.THierarchicalCostFunctionProvider_SquaredEuclidean(
            MultiCenter,MultiX)
    # eps scaling
    epsScalingHandler=MultiScaleOT.TEpsScalingHandler()
    epsScalingHandler.setupGeometricMultiLayerB(nLayers,1.,4.,2,2)

    # error goal
    errorGoal=1E-3
    # sinkhorn solver object
    SinkhornSolver=MultiScaleOT.TSinkhornSolverStandard(epsScalingHandler,
            0,hierarchyDepth,errorGoal,
            MultiCenter,MultiX,costFunction
            )

    SinkhornSolver.initialize()
    #print(SinkhornSolver.solve())
    SinkhornSolver.solve()
    #print("solving time: ",t2-t1)

    kernelData=SinkhornSolver.getKernelCSRDataTuple()
    xres=len(muRef)
    yres=len(mu)
    kernel=scipy.sparse.csr_array(kernelData,shape=(xres,yres))

    tan=LinOT.LogW2(kernel,posRef,pos)
    return tan

In [ ]:
filename_tan = save_dir / f"{structure_name_folder}_tan.npz"

# Step 1: Initial creation (compute and save index 0 if the file doesn't exist yet)
if not filename_tan.is_file():
    tan = getTan(0)
    np.savez(filename_tan, tan0=tan)

# Step 2: Loop through all indices and incrementally add any missing tangent components
if filename_tan.is_file():
    for i in range(nImg):
        tanfile = np.load(filename_tan)
        print(f"{i + 1}/{nImg}")
        # Calculate and append only if this component hasn't been saved yet
        if f"tan{i}" not in tanfile.keys():
            tan = getTan(i)
            np.savez(filename_tan, **tanfile, **{f"tan{i}": tan})